In [1]:
import pandas as pd
pd.options.display.float_format = '{:.4f}'.format
import numpy as np

# Data loading

In [2]:
rfr_daily = 0.00 
EVAL_START = pd.Timestamp("2015-01-01")
EVAL_END   = pd.Timestamp("2025-12-31")

START_DATE = pd.Timestamp("1995-01-01") #garde fou 

In [3]:
#pre run
PARQUET_DIR = "data_coche/19952025/parquets_by_field"   


close = pd.read_parquet(f"{PARQUET_DIR}/close_price_coche9525.parquet", engine = "fastparquet")
open_ = pd.read_parquet(f"{PARQUET_DIR}/open_coche9525.parquet", engine = "fastparquet")
high = pd.read_parquet(f"{PARQUET_DIR}/high_coche9525.parquet", engine = "fastparquet")
low = pd.read_parquet(f"{PARQUET_DIR}/low_coche9525.parquet", engine = "fastparquet")
volume = pd.read_parquet(f"{PARQUET_DIR}/volume_coche9525.parquet", engine = "fastparquet")
vwap = pd.read_parquet(f"{PARQUET_DIR}/vwap_coche9525.parquet", engine = "fastparquet")
mktcap = pd.read_parquet(f"{PARQUET_DIR}/market_cap_coche9525.parquet", engine = "fastparquet")

# datetime au cas ou mais ok 
close.index  = pd.to_datetime(close.index, errors="coerce")
open_.index  = pd.to_datetime(open_.index, errors="coerce")
high.index   = pd.to_datetime(high.index, errors="coerce")
low.index    = pd.to_datetime(low.index, errors="coerce")
volume.index = pd.to_datetime(volume.index, errors="coerce")
vwap.index   = pd.to_datetime(vwap.index, errors="coerce")
mktcap.index   = pd.to_datetime(mktcap.index, errors="coerce")

close  = close.apply(pd.to_numeric, errors="coerce")
open_  = open_.apply(pd.to_numeric, errors="coerce")
high   = high.apply(pd.to_numeric, errors="coerce")
low    = low.apply(pd.to_numeric, errors="coerce")
volume = volume.apply(pd.to_numeric, errors="coerce")
vwap   = vwap.apply(pd.to_numeric, errors="coerce")
mktcap = mktcap.apply(pd.to_numeric, errors="coerce")


# filter les dates pcq garde fou 

close  = close.loc[close.index >= START_DATE].sort_index()
open_  = open_.loc[open_.index >= START_DATE].sort_index()
high   = high.loc[high.index >= START_DATE].sort_index()
low    = low.loc[low.index >= START_DATE].sort_index()
volume = volume.loc[volume.index >= START_DATE].sort_index()
vwap   = vwap.loc[vwap.index >= START_DATE].sort_index()
mktcap = mktcap.loc[mktcap.index >= START_DATE].sort_index()

FEATURES = {
    "close": close,
    "open": open_,
    "high": high,
    "low": low,
    "vwap": vwap,
    "volume": volume,
    "mcap": mktcap,   
}



# Construction des dfs

## Returns normaux

In [4]:
returns =  close / close.shift(1) - 1
returns

ticker,0945329D UQ Equity,0945329D US Equity,0964591D UQ Equity,0964591D US Equity,0964591D UW Equity,1040983D UQ Equity,1040983D US Equity,1255459D UQ Equity,1255459D US Equity,1255459D UW Equity,...,UAL UW Equity,ULTA UW Equity,VEON UW Equity,VIAB UW Equity,VRSK UW Equity,WBD UW Equity,WCRX UW Equity,WDAY UW Equity,ZM UW Equity,ZS UW Equity
DATES,,,,,,,,,,,,,,,,,,,,,
1995-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-04,-0.1765,-0.1765,0.0561,0.0561,0.0561,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-05,-0.1429,-0.1429,0.0442,0.0442,0.0442,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-06,0.0000,0.0000,-0.0085,-0.0085,-0.0085,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-09,-0.1667,-0.1667,-0.0085,-0.0085,-0.0085,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0101,0.0065,-0.0153,NaN,-0.0025,0.0058,NaN,0.0025,0.0049,0.0063
2025-12-22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0178,0.0252,0.0233,NaN,0.0111,0.0353,NaN,-0.0029,0.0019,-0.0022
2025-12-23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.0215,-0.0145,-0.0031,NaN,-0.0090,0.0139,NaN,-0.0117,-0.0314,-0.0118


In [5]:
returns.to_parquet("MAIN_DATA/returns.parquet", engine="fastparquet")

In [6]:
forward_returns = close.shift(-1) / close - 1.0
forward_returns.to_parquet("MAIN_DATA/forward_returns.parquet", engine="fastparquet")
# pas besoin, modif 

# Résidualizers 

In [7]:
universe = ~close.isna()


def standardize_alphas(alpha_df: pd.DataFrame, universe: pd.DataFrame):
    """
    Convert alphas into investable positions using cross-sectional ranking.
    
    This replaces z-score normalization with:
    - ranking (robust to outliers)
    - centering (dollar-neutral)
    - scaling (controlled leverage)
    
    Args:
        alpha_df: Alpha signals DataFrame (dates x assets)
    
    Returns:
        positions: Dollar-neutral portfolio weights
    """

    # Only keep alpha inside of the universe
    alpha_df = alpha_df.where(universe)

    # 1. Rank cross-sectionally (0 to 1)
    ranks = alpha_df.rank(axis=1, pct=True)
    
    # 2. Center to [-0.5, 0.5] → long/short
    centered = ranks - 0.5
    
    # 3. Normalize leverage (L1 norm = 1)
    denom = centered.abs().sum(axis=1)
    denom = denom.replace(0, np.nan)
    
    positions = centered.div(denom, axis=0)
    
    # 4. Clean
    positions = positions.replace([np.inf, -np.inf], np.nan)
    positions = positions.fillna(0.0)

    # Only keep positions inside of the universe
    positions = positions.where(universe)
    
    return positions
  # vol a calc avantt

def normalize_alpha_by_volatility(alpha_df: pd.DataFrame, returns_normaux = pd.DataFrame, window=63):
    """
    Normalize alpha by rolling volatility.
    
    Args:
        alpha_df: Raw alpha signals DataFrame
        window: Rolling window for volatility calculation (default 63 days ~ 3 months)
    
    Returns:
        Normalized alpha DataFrame (alpha / rolling_volatility)
    """
    rolling_vol = returns_normaux.rolling(
        window=window, 
        min_periods=max(2, window // 2)
    ).std()
    ## faire les returns du stock 63d
    rolling_vol = rolling_vol.clip(lower=1e-8) # chgt 
    normalized_alpha = alpha_df / rolling_vol
    normalized_alpha = normalized_alpha.replace([np.inf, -np.inf], np.nan)
    normalized_alpha = normalized_alpha.fillna(0.0) #voir si ca fait pas nimp 
    return normalized_alpha

Momentum

In [8]:
df_momentum= returns.rolling(window = 21*11, min_periods=63).mean().shift(21)
df_momentum  =standardize_alphas(df_momentum, universe)
df_momentum = df_momentum.ffill(limit = 21)
df_momentum
# df_momentum.to_parquet("MAIN_DATA/df_momentum.parquet", engine="fastparquet")

ticker,0945329D UQ Equity,0945329D US Equity,0964591D UQ Equity,0964591D US Equity,0964591D UW Equity,1040983D UQ Equity,1040983D US Equity,1255459D UQ Equity,1255459D US Equity,1255459D UW Equity,...,UAL UW Equity,ULTA UW Equity,VEON UW Equity,VIAB UW Equity,VRSK UW Equity,WBD UW Equity,WCRX UW Equity,WDAY UW Equity,ZM UW Equity,ZS UW Equity
DATES,,,,,,,,,,,,,,,,,,,,,
1995-01-03,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-04,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-05,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-06,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-09,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0006,0.0019,0.0060,NaN,-0.0057,0.0073,NaN,-0.0051,-0.0021,0.0054
2025-12-22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0004,0.0064,0.0148,NaN,-0.0156,0.0184,NaN,-0.0132,-0.0060,0.0144
2025-12-23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.0011,0.0062,0.0147,NaN,-0.0159,0.0189,NaN,-0.0117,-0.0087,0.0125


Reversal

In [9]:
df_reversal_21 = - returns.rolling(window=21, min_periods=21//2).mean()
df_reversal_21 = standardize_alphas(df_reversal_21, universe) 
df_reversal_21 = df_reversal_21.ffill(limit = 3)
df_reversal_21.to_parquet("MAIN_DATA/df_reversal_21.parquet", engine="fastparquet")

In [10]:
df_reversal_5 = - returns.rolling(window=5, min_periods=5//2).mean()
df_reversal_5 = standardize_alphas(df_reversal_5, universe) 
df_reversal_5 = df_reversal_5.ffill( limit = 1)
df_reversal_5.to_parquet("MAIN_DATA/df_reversal_5.parquet", engine="fastparquet")

df_reversal_2= - returns.rolling(window=2, min_periods=2//2).mean()
df_reversal_2 = standardize_alphas(df_reversal_2, universe) 
df_reversal_2.to_parquet("MAIN_DATA/df_reversal_2.parquet", engine="fastparquet")


df_reversal_1= - returns.rolling(window=1, min_periods=1).mean()
df_reversal_1 = standardize_alphas(df_reversal_1, universe) 
df_reversal_1.to_parquet("MAIN_DATA/df_reversal_1.parquet", engine="fastparquet")

Long term reversal



In [11]:
df_long_term_reversal =  - returns.rolling(window = 252*7, min_periods = 252*4).mean().shift(252*4)
df_long_term_reversal = standardize_alphas(df_long_term_reversal, universe) 
df_long_term_reversal = df_long_term_reversal.ffill(limit = 21)
df_long_term_reversal.to_parquet("MAIN_DATA/df_long_term_reversal.parquet", engine="fastparquet")

Low vol : 

In [12]:
df_low_volatility = -  returns.rolling(window=252*2, min_periods=63).std()
df_low_volatility = standardize_alphas(df_low_volatility, universe) 
df_low_volatility = df_low_volatility.ffill(limit= 21)
df_low_volatility.to_parquet("MAIN_DATA/df_low_volatility.parquet", engine="fastparquet")

In [13]:
df_size = - np.log(mktcap)  
df_size = standardize_alphas(df_size, universe) 
df_size = df_size.ffill(limit = 21)
df_size.to_parquet("MAIN_DATA/df_size.parquet", engine="fastparquet")
### 

# Adjustment factor cleanup 

In [14]:
adj_factor_path = 'adj_factor_df.xlsx'

In [19]:

EXCEL_PATH = 'adj_factor_df.xlsx'          
OUTPUT_PATH = "stacked_data_adj_factor.parquet"

# Tabs à exclure
EXCLUDE_TABS = ["Final Template"]
 
# Colonnes à extraire : C (index 2) → UA (index 366 si UA = colonne 367)
# Colonne B = index (date), colonnes C:UA = données
# En Excel : A=0, B=1, C=2, ..., UA = ?
# U=20, A=0 → UA = 20*26 + 0 = col 520 ? Non.
# En fait : UA en notation Excel :
#   U = 21e lettre, A = 1e lettre → UA = 21*26 + 1 - 1 = 547 ? 
#   Formule: col_index("UA") = (ord('U')-ord('A')+1)*26 + (ord('A')-ord('A')+1) - 1
#   = 21*26 + 1 - 1 = 546
# Donc usecols = [1] + list(range(2, 547))  → col B (date) + C à UA
# ──────────────────────────────────────────────────────────────────────────────
 
def col_letter_to_index(col: str) -> int:
    """Convert Excel column letter(s) to 0-based index. E.g. 'A'->0, 'UA'->546"""
    col = col.upper()
    result = 0
    for char in col:
        result = result * 26 + (ord(char) - ord('A') + 1)
    return result - 1
 
DATE_COL_IDX  = col_letter_to_index("C")   # = 2
START_COL_IDX = col_letter_to_index("D")   # = 3
END_COL_IDX   = col_letter_to_index("UA")  # = 546
 
print(f"Date col index : {DATE_COL_IDX}")
print(f"Data cols      : {START_COL_IDX} → {END_COL_IDX}")
 
# ── LECTURE ────────────────────────────────────────────────────────────────────
xl = pd.ExcelFile(EXCEL_PATH)
all_sheets = [s for s in xl.sheet_names if s not in EXCLUDE_TABS]
print(f"\nTabs à traiter ({len(all_sheets)}) : {all_sheets}")
 
all_dfs = []
 
for sheet in all_sheets:
    print(f"  → Lecture de '{sheet}'...")
    df = pd.read_excel(
        EXCEL_PATH,
        sheet_name=sheet,
        header=0,
        index_col = None,
        usecols="C:UA",  # B à UA
    )
 
    # La première colonne lue est la date (colonne B)
    date_col_name = df.columns[0]
    df = df.rename(columns={date_col_name: "date"})

    seen = {}
    new_cols = []
    for col in df.columns:
        base = col.rsplit(".", 1)[0] if col != "date" and "." in str(col) and str(col).rsplit(".", 1)[-1].isdigit() else col
        if base not in seen:
            seen[base] = 0
            new_cols.append(base)
        else:
            seen[base] += 1
            new_cols.append(f"{base}_dup{seen[base]}")  # marque les vrais doublons
    df.columns = new_cols

    df = df[[c for c in df.columns if "_dup" not in str(c)]]
 
 
    # Supprimer les lignes sans date
    df = df.dropna(subset=["date"])
 
    # Convertir la date en datetime si ce n'est pas déjà le cas
    df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")
    df = df.dropna(subset=["date"])
 
    # Ajouter une colonne indiquant la source
    # df["source_tab"] = sheet
 
    all_dfs.append(df)
 
# ── STACK ──────────────────────────────────────────────────────────────────────
print("\nAssemblage de tous les tabs...")
stacked = pd.concat(all_dfs, ignore_index=True)
 
# Mettre la date comme index
stacked = stacked.set_index("date").sort_index()
# stacked.rename({'0964591D UQ Equity.1' : '0964591D UQ Equity'}, inplace=True)
 
print(f"\nShape finale : {stacked.shape}")
print(f"Période      : {stacked.index.min()} → {stacked.index.max()}")
print(f"Colonnes     : {list(stacked.columns[:5])} ... {list(stacked.columns[-3:])}")
 


Date col index : 2
Data cols      : 3 → 546

Tabs à traiter (6) : ['1995-2000', '2000-2005', '2005-2010', '2010-2015', '2015-2020', '2020-02']
  → Lecture de '1995-2000'...
  → Lecture de '2000-2005'...
  → Lecture de '2005-2010'...
  → Lecture de '2010-2015'...
  → Lecture de '2015-2020'...
  → Lecture de '2020-02'...

Assemblage de tous les tabs...

Shape finale : (11330, 544)
Période      : 1995-01-01 00:00:00 → 2025-12-31 00:00:00
Colonnes     : ['0964591D UQ Equity', '1040983D UQ Equity', '1518855D US Equity', '1519128D UQ Equity', '1541931D UQ Equity'] ... ['TRI UN Equity', 'TRI UW Equity', 'SOLS UW Equity']


In [20]:
stacked = stacked.ffill(limit = 21)

In [21]:
stacked

,0964591D UQ Equity,1040983D UQ Equity,1518855D US Equity,1519128D UQ Equity,1541931D UQ Equity,1579957D UQ Equity,166783Q UQ Equity,1683351D UQ Equity,1683997D UQ Equity,1746513D UQ Equity,...,SMCI UW Equity,APP UW Equity,AXON UW Equity,MSTR UW Equity,PLTR UW Equity,SHOP UN Equity,SHOP UW Equity,TRI UN Equity,TRI UW Equity,SOLS UW Equity
date,,,,,,,,,,,,,,,,,,,,,
1995-01-01,14.0000,NaN,NaN,36.0000,41.8750,NaN,NaN,80.3750,41.0000,50.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-02,14.0000,NaN,NaN,36.0000,41.8750,NaN,NaN,80.3750,41.0000,50.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-03,13.3750,NaN,NaN,34.8750,40.7500,NaN,NaN,80.3750,39.8750,48.7500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-04,14.1250,NaN,NaN,35.2500,40.5000,NaN,NaN,61.7500,42.2500,48.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-05,14.7500,NaN,NaN,35.7500,40.5000,NaN,NaN,62.0000,43.6250,47.5000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,30.6400,714.2300,583.8400,158.8100,188.7100,170.8600,170.8300,133.1500,133.2100,49.7700
2025-12-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,30.6400,714.2300,583.8400,158.8100,188.7100,170.8600,170.8300,133.1500,133.2100,49.7700
2025-12-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,30.0800,698.8200,580.5500,155.3900,184.1800,167.8200,167.8800,133.1600,133.2200,49.2900


In [22]:
stacked  =standardize_alphas(stacked, universe)
stacked = stacked.ffill(limit = 21)

In [26]:

stacked.to_parquet('MAIN_DATA/stacked_data_adj_factor.parquet')
# print(f"\n Fichier sauvegardé : {'MAIN_DATA/stacked_data_adj_factor.parquet'}")

In [27]:
stacked

,0964591D UQ Equity,1040983D UQ Equity,1518855D US Equity,1519128D UQ Equity,1541931D UQ Equity,1579957D UQ Equity,166783Q UQ Equity,1683351D UQ Equity,1683997D UQ Equity,1746513D UQ Equity,...,SMCI UW Equity,APP UW Equity,AXON UW Equity,MSTR UW Equity,PLTR UW Equity,SHOP UN Equity,SHOP UW Equity,TRI UN Equity,TRI UW Equity,SOLS UW Equity
date,,,,,,,,,,,,,,,,,,,,,
1995-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-03,-0.0044,NaN,NaN,0.0028,0.0050,NaN,NaN,0.0081,0.0044,0.0064,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-04,-0.0040,NaN,NaN,0.0033,0.0047,NaN,NaN,0.0076,0.0050,0.0061,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-05,-0.0039,NaN,NaN,0.0033,0.0047,NaN,NaN,0.0077,0.0052,0.0064,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.0151,0.0202,0.0177,0.0007,0.0062,0.0028,0.0032,-0.0001,-0.0002,-0.0113
2025-12-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.0151,0.0202,0.0177,0.0007,0.0062,0.0028,0.0032,-0.0001,-0.0002,-0.0113
2025-12-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.0151,0.0202,0.0177,0.0007,0.0062,0.0028,0.0032,-0.0001,-0.0002,-0.0113
